In [16]:
import pandas as pd
import numpy as np

# ============================================================
# REAL ESTATE RWA UNDERWRITING & TOKENIZATION SCREENING
# ============================================================
#
# PURPOSE:
# This proof-of-concept combines:
# - Real estate financial analysis
# - Geography / regional planning concepts
# - Geospatial risk variables
# - Risk-adjusted valuation
# - Tokenization modelling
# - RWA screening logic
#
# IMPORTANT:
# All property values, rental figures, coordinates and risk
# scores used here are synthetic/illustrative data created
# for research and technical demonstration purposes.
#
# This notebook does NOT perform:
# - Legal title verification
# - Regulatory approval
# - Property certification
# - Actual blockchain token issuance
# - Investment recommendation
# ============================================================


# ============================================================
# 1. BUILD SYNTHETIC PROPERTY DATASET
# ============================================================

prop_ids = [
    "PROP-001",
    "PROP-002",
    "PROP-003",
    "PROP-004",
    "PROP-005"
]

locations = [
    "Airport Residential, Accra",
    "Cape Coast (Near UCC)",
    "Kumasi Asokwa",
    "East Legon, Accra",
    "Elmina Coastline"
]

market_values = [
    5_000_000,
    1_200_000,
    2_500_000,
    4_800_000,
    950_000
]

annual_rentals = [
    350_000,
    84_000,
    180_000,
    320_000,
    76_000
]

# Prototype risk score:
# 1 = lower risk
# 5 = higher risk
risk_scores = [
    1,
    2,
    1,
    2,
    5
]

# Geographic coordinates
latitudes = [
    5.606,
    5.114,
    6.674,
    5.632,
    5.085
]

longitudes = [
    -0.168,
    -1.282,
    -1.602,
    -0.151,
    -1.348
]


# ============================================================
# 2. CREATE DATAFRAME
# ============================================================

raw_listings = pd.DataFrame({
    "Property_ID": prop_ids,
    "Location": locations,
    "Market_Value_GHS": market_values,
    "Annual_Rental_Income_GHS": annual_rentals,
    "Zoning_Risk_Score": risk_scores,
    "Latitude": latitudes,
    "Longitude": longitudes
})


# ============================================================
# 3. BASIC DATA VALIDATION
# ============================================================

required_columns = [
    "Property_ID",
    "Location",
    "Market_Value_GHS",
    "Annual_Rental_Income_GHS",
    "Zoning_Risk_Score",
    "Latitude",
    "Longitude"
]

missing_columns = [
    col for col in required_columns
    if col not in raw_listings.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if (raw_listings["Market_Value_GHS"] <= 0).any():
    raise ValueError(
        "Market values must be greater than zero."
    )

if (raw_listings["Annual_Rental_Income_GHS"] < 0).any():
    raise ValueError(
        "Annual rental income cannot be negative."
    )

if not raw_listings["Zoning_Risk_Score"].between(1, 5).all():
    raise ValueError(
        "Zoning_Risk_Score must be between 1 and 5."
    )


# ============================================================
# 4. MODEL PARAMETERS
# ============================================================

# Assets with a score >= 4 fail the prototype screening rule.
HIGH_RISK_THRESHOLD = 4

# Illustrative modelling assumption:
# high-risk properties receive a 20% valuation haircut.
RISK_HAIRCUT_PCT = 20

# Theoretical token denomination for this prototype.
TOKEN_PRICE_GHS = 100


# ============================================================
# 5. RWA UNDERWRITING & SCREENING FUNCTION
# ============================================================

def process_rwa_pipeline(df):
    """
    Processes property financial metrics and performs
    prototype RWA screening.

    IMPORTANT:
    - Risk scores are illustrative.
    - The 20% haircut is a modelling assumption.
    - Token quantities are theoretical/modelled values.
    - Screening results are NOT legal or regulatory approvals.
    """

    result = df.copy()

    # --------------------------------------------------------
    # A. GROSS RENTAL YIELD
    # --------------------------------------------------------

    result["Gross_Yield_Pct"] = (
        result["Annual_Rental_Income_GHS"]
        / result["Market_Value_GHS"]
    ) * 100


    # --------------------------------------------------------
    # B. RISK ADJUSTMENT
    # --------------------------------------------------------

    result["Risk_Adjustment_Pct"] = np.where(
        result["Zoning_Risk_Score"] >= HIGH_RISK_THRESHOLD,
        RISK_HAIRCUT_PCT,
        0
    )


    # --------------------------------------------------------
    # C. RISK-ADJUSTED PROPERTY VALUE
    # --------------------------------------------------------

    result["Risk_Adjusted_Value_GHS"] = (
        result["Market_Value_GHS"]
        * (
            1
            - result["Risk_Adjustment_Pct"] / 100
        )
    )


    # --------------------------------------------------------
    # D. MODELLED TOKEN QUANTITY
    # --------------------------------------------------------
    #
    # This is a theoretical quantity based on the modelled
    # property value and token denomination.
    #
    # It does NOT represent actual blockchain tokens.
    # --------------------------------------------------------

    result["Modelled_Token_Quantity"] = (
        result["Risk_Adjusted_Value_GHS"]
        / TOKEN_PRICE_GHS
    ).astype(int)


    # --------------------------------------------------------
    # E. RWA SCREENING STATUS
    # --------------------------------------------------------

    result["RWA_Screening_Status"] = np.where(
        result["Zoning_Risk_Score"] >= HIGH_RISK_THRESHOLD,
        "REJECTED - HIGH RISK",
        "ELIGIBLE FOR FURTHER TOKENIZATION REVIEW"
    )


    # --------------------------------------------------------
    # F. ELIGIBLE TOKEN QUANTITY
    # --------------------------------------------------------
    #
    # Rejected assets receive zero eligible token quantity.
    # --------------------------------------------------------

    result["Eligible_Token_Quantity"] = np.where(
        result["Zoning_Risk_Score"] >= HIGH_RISK_THRESHOLD,
        0,
        result["Modelled_Token_Quantity"]
    ).astype(int)


    # --------------------------------------------------------
    # G. SCREENING REASON
    # --------------------------------------------------------

    result["Screening_Reason"] = np.where(
        result["Zoning_Risk_Score"] >= HIGH_RISK_THRESHOLD,
        "Failed prototype high-risk screening threshold",
        "Passed prototype risk screening"
    )

    return result


# ============================================================
# 6. RUN THE PIPELINE
# ============================================================

processed_rwa = process_rwa_pipeline(raw_listings)


# ============================================================
# 7. SAVE PROCESSED DATASET
# ============================================================

processed_rwa.to_csv(
    "rwa_tokenized_portfolio.csv",
    index=False
)


# ============================================================
# 8. PREPARE PORTFOLIO SCREENING MATRIX
# ============================================================

display_columns = [
    "Property_ID",
    "Location",
    "Market_Value_GHS",
    "Gross_Yield_Pct",
    "Zoning_Risk_Score",
    "Risk_Adjustment_Pct",
    "Risk_Adjusted_Value_GHS",
    "Modelled_Token_Quantity",
    "Eligible_Token_Quantity",
    "RWA_Screening_Status",
    "Screening_Reason"
]

display_df = processed_rwa[display_columns].copy()


# ============================================================
# 9. FORMAT RESULTS FOR PRESENTATION
# ============================================================

display_df["Gross_Yield_Pct"] = (
    display_df["Gross_Yield_Pct"].round(2)
)

display_df["Market_Value_GHS"] = (
    display_df["Market_Value_GHS"].astype(int)
)

display_df["Risk_Adjusted_Value_GHS"] = (
    display_df["Risk_Adjusted_Value_GHS"].astype(int)
)


# ============================================================
# 10. DISPLAY FINAL RWA SCREENING MATRIX
# ============================================================

display_df

,Property_ID,Location,Market_Value_GHS,Gross_Yield_Pct,Zoning_Risk_Score,Risk_Adjustment_Pct,Risk_Adjusted_Value_GHS,Modelled_Token_Quantity,Eligible_Token_Quantity,RWA_Screening_Status,Screening_Reason
0,PROP-001,"Airport Residential, Accra",5000000,7.00,1,0,5000000,50000,50000,ELIGIBLE FOR FURTHER TOKENIZATION REVIEW,Passed prototype risk screening
1,PROP-002,Cape Coast (Near UCC),1200000,7.00,2,0,1200000,12000,12000,ELIGIBLE FOR FURTHER TOKENIZATION REVIEW,Passed prototype risk screening
2,PROP-003,Kumasi Asokwa,2500000,7.20,1,0,2500000,25000,25000,ELIGIBLE FOR FURTHER TOKENIZATION REVIEW,Passed prototype risk screening
3,PROP-004,"East Legon, Accra",4800000,6.67,2,0,4800000,48000,48000,ELIGIBLE FOR FURTHER TOKENIZATION REVIEW,Passed prototype risk screening
4,PROP-005,Elmina Coastline,950000,8.00,5,20,760000,7600,0,REJECTED - HIGH RISK,Failed prototype high-risk screening threshold
